# Caso de clase: Minería de datos exploratoria para Amazon Singapore

## De la base de datos a la decisión gerencial

Este notebook desarrolla un proceso de **minería de datos exploratoria** usando la base `Amazon.xlsx`.  
La intención **no es construir modelos de Machine Learning todavía**, sino preparar una lectura gerencial clara mediante un EDA completo: limpieza, diagnóstico de calidad, visualización, comparación competitiva y generación de hipótesis estratégicas.

La lógica sigue la presentación **El plano de navegación: de la gran data a la estrategia clara**:

- Primero se define el **norte del negocio**.
- Luego se traduce la base a reglas de procesamiento.
- Después se diagnostican faltantes y atípicos.
- Finalmente se construyen visualizaciones que permitan pasar de datos crudos a conocimiento estratégico.

---

## Pregunta gerencial del caso

> ¿Qué nos dicen los datos de experiencia de cliente sobre la posición competitiva de Amazon frente a otros comercios electrónicos, y cuáles dimensiones de la experiencia deben priorizarse antes de pensar en modelos predictivos?


## 0. Librerías y configuración visual

Usaremos `pandas` para manipulación de datos, `matplotlib` y `seaborn` para visualización.  
La paleta se mantiene sobria para facilitar su uso en clase y en presentaciones gerenciales.


In [ ]:
# ============================================================
# 0. LIBRERÍAS Y CONFIGURACIÓN
# ============================================================

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Configuración visual
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.figsize"] = (10, 5)
plt.rcParams["axes.titlesize"] = 14
plt.rcParams["axes.labelsize"] = 11
plt.rcParams["xtick.labelsize"] = 9
plt.rcParams["ytick.labelsize"] = 9
plt.rcParams["figure.dpi"] = 120

DATA_PATH = Path("Amazon.xlsx")

if not DATA_PATH.exists():
    raise FileNotFoundError(
        "No se encuentra Amazon.xlsx. Coloque el archivo en la misma carpeta del notebook."
    )

print("Librerías cargadas correctamente.")

## 1. Carga de datos y diccionario

La minería de datos no inicia con un algoritmo, sino con una comprensión mínima del archivo: hojas disponibles, tamaño de la base, variables y significado de los códigos.


In [ ]:
# ============================================================
# 1. CARGA DE DATOS
# ============================================================

xls = pd.ExcelFile(DATA_PATH)
print("Hojas disponibles:", xls.sheet_names)

raw = pd.read_excel(DATA_PATH, sheet_name="Raw Data")
dict_raw = pd.read_excel(DATA_PATH, sheet_name="Variable Dictionary", header=None)

print("Dimensión de la base cruda:", raw.shape)
display(raw.head())

In [ ]:
# Diccionario de variables en formato limpio
var_dict = dict_raw.iloc[1:].copy()
var_dict.columns = ["variable", "description", "range", "values_key", "missing_codes"]
var_dict = var_dict.iloc[1:].reset_index(drop=True)

# Vista compacta del diccionario
pd.set_option("display.max_colwidth", 90)
display(var_dict.head(15))

## 2. Norte del negocio y mapa de variables

Para evitar una exploración sin dirección, agruparemos las variables según dimensiones de experiencia de cliente.  
Esto permite convertir muchas columnas en una lectura estratégica: producto, navegación, precio, confianza, entrega, comunicación y perfil del cliente.


In [ ]:
# ============================================================
# 2. MAPA DE VARIABLES GERENCIALES
# ============================================================

# Variables principales de evaluación global
core_vars = ["poverq", "soverq", "pq", "satis", "repur", "recomm", "Q19", "VN_1009_Q20A"]

# Variables de experiencia específicas
experience_vars = [f"VN_1009_TP{str(i).zfill(2)}" for i in range(1, 20)]

# Variables transaccionales y comportamiento
behavior_vars = ["VN_1009_TP20", "VN_1009_TP21", "VN_1009_TP24_1", "VN_1009_TP24_2",
                 "VN_1009_TP22", "VN_1009_TP23", "Q9C_P", "Q9D", "VN_1009_TP25A"]

# Variables demográficas
demo_vars = ["age", "race", "work", "pincome", "income", "educat", "childsupp",
             "marital", "gender", "house", "DOI"]

# Agrupación conceptual para análisis gerencial
experience_dimensions = {
    "Oferta y producto": ["VN_1009_TP01", "VN_1009_TP02", "VN_1009_TP05", "VN_1009_TP17"],
    "Navegación y búsqueda": ["VN_1009_TP03", "VN_1009_TP04", "VN_1009_TP08", "VN_1009_TP10"],
    "Precio y promociones": ["pq", "VN_1009_TP06"],
    "Información y confianza": ["VN_1009_TP07", "VN_1009_TP12", "VN_1009_TP13"],
    "Entrega": ["VN_1009_TP14", "VN_1009_TP15", "VN_1009_TP16"],
    "Comunicación y posventa": ["VN_1009_TP18", "VN_1009_TP19", "VN_1009_TP22", "VN_1009_TP23"],
    "Resultados de cliente": ["satis", "repur", "recomm", "VN_1009_Q20A"]
}

summary_map = pd.DataFrame({
    "Dimensión": list(experience_dimensions.keys()),
    "Variables": [", ".join(v) for v in experience_dimensions.values()],
    "Número de variables": [len(v) for v in experience_dimensions.values()]
})

display(summary_map)

## 3. Diagnóstico inicial de estructura

Antes de limpiar, observamos tipos de variables, número de valores únicos y posibles códigos especiales.  
En la presentación esto corresponde a traducir el negocio a reglas de procesamiento: no se borra ni transforma sin entender qué significa cada código.


In [ ]:
# ============================================================
# 3. DIAGNÓSTICO INICIAL DE ESTRUCTURA
# ============================================================

structure = pd.DataFrame({
    "variable": raw.columns,
    "dtype": raw.dtypes.astype(str).values,
    "non_null": raw.notna().sum().values,
    "nulls": raw.isna().sum().values,
    "unique_values": raw.nunique(dropna=False).values
})

# Proporción de registros por compañía
company_counts = raw["company_v"].value_counts().reset_index()
company_counts.columns = ["company", "n"]
company_counts["share"] = company_counts["n"] / company_counts["n"].sum()

display(structure.head(20))
display(company_counts)

In [ ]:
# Gráfico 1: tamaño de muestra por compañía
plt.figure(figsize=(10, 5))
sns.barplot(data=company_counts, x="company", y="n")
plt.title("Tamaño de muestra por compañía evaluada")
plt.xlabel("Compañía")
plt.ylabel("Número de encuestados")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

## 4. Reglas de limpieza: códigos 97, 98, 99 y 999

El diccionario indica que varias variables usan códigos como `97`, `98`, `99` o `999` para respuestas no válidas, no sabe/no responde o rechazos.  
Estos valores no deben tratarse como números reales de satisfacción o gasto.

En esta etapa no estamos “mejorando” artificialmente la base; estamos evitando que códigos administrativos contaminen la lectura gerencial.


In [ ]:
# ============================================================
# 4. LIMPIEZA DE CÓDIGOS ESPECIALES
# ============================================================

df = raw.copy()

# Columnas numéricas candidatas a contener códigos 97, 98, 99
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()

# Reemplazar códigos especiales según diccionario del caso
special_missing = [97, 98, 99]
for col in numeric_cols:
    df[col] = df[col].replace(special_missing, np.nan)

# Q9D usa 999 como código especial según diccionario
if "Q9D" in df.columns:
    df["Q9D"] = df["Q9D"].replace(999, np.nan)

# DOI a formato fecha
if "DOI" in df.columns:
    df["DOI"] = pd.to_datetime(df["DOI"], errors="coerce")

print("Valores especiales convertidos a NaN.")
print("Dimensión después de limpieza:", df.shape)
display(df.head())

## 5. Diagnóstico de incompletitud

La presentación distingue entre faltantes aleatorios, sistemáticos y ocultos.  
Aquí no vamos a imputar todavía; primero debemos observar dónde están los faltantes y si pueden afectar la interpretación.


In [ ]:
# ============================================================
# 5. DIAGNÓSTICO DE FALTANTES
# ============================================================

missing = pd.DataFrame({
    "variable": df.columns,
    "missing_n": df.isna().sum().values,
    "missing_pct": (df.isna().mean().values * 100).round(2)
}).sort_values("missing_pct", ascending=False)

missing_nonzero = missing[missing["missing_n"] > 0]

display(missing_nonzero.head(20))

In [ ]:
# Gráfico 2: variables con mayor porcentaje de faltantes
plt.figure(figsize=(10, 6))
top_missing = missing_nonzero.head(20)
sns.barplot(data=top_missing, y="variable", x="missing_pct")
plt.title("Variables con mayor proporción de datos faltantes")
plt.xlabel("Porcentaje de faltantes")
plt.ylabel("Variable")
plt.tight_layout()
plt.show()

In [ ]:
# Gráfico 3: mapa de calor de faltantes para variables con faltantes
cols_missing = missing_nonzero["variable"].head(25).tolist()

if len(cols_missing) > 0:
    plt.figure(figsize=(12, 6))
    sns.heatmap(df[cols_missing].isna(), cbar=False)
    plt.title("Mapa de faltantes en variables críticas")
    plt.xlabel("Variable")
    plt.ylabel("Registro")
    plt.tight_layout()
    plt.show()
else:
    print("No hay variables con faltantes después de la limpieza.")

## 6. Distribución de las variables principales

Primero observamos los indicadores globales: calidad del producto, calidad del servicio, precio percibido, satisfacción, recompra y recomendación.  
Esta lectura permite identificar si el caso está dominado por clientes satisfechos, insatisfechos o por una distribución heterogénea.


In [ ]:
# ============================================================
# 6. DISTRIBUCIÓN DE VARIABLES PRINCIPALES
# ============================================================

available_core = [c for c in core_vars if c in df.columns]

# Tabla descriptiva
core_desc = df[available_core].describe().T
core_desc["missing_pct"] = df[available_core].isna().mean() * 100
core_desc = core_desc.round(2)
display(core_desc)

In [ ]:
# Gráfico 4: histogramas de variables principales
for col in available_core:
    plt.figure(figsize=(8, 4))
    sns.histplot(df[col], bins=10, kde=False)
    plt.title(f"Distribución de {col}")
    plt.xlabel(col)
    plt.ylabel("Frecuencia")
    plt.tight_layout()
    plt.show()

In [ ]:
# Gráfico 5: boxplots de variables principales
plt.figure(figsize=(12, 5))
sns.boxplot(data=df[available_core], orient="h")
plt.title("Distribución comparada de indicadores globales")
plt.xlabel("Escala de evaluación")
plt.ylabel("Variable")
plt.tight_layout()
plt.show()

## 7. Amazon frente al mercado

Ahora comparamos Amazon con el conjunto de competidores.  
La idea no es declarar causalidad, sino identificar brechas visibles que puedan orientar preguntas estratégicas.


In [ ]:
# ============================================================
# 7. AMAZON VS MERCADO
# ============================================================

df["is_amazon"] = np.where(df["company_v"].astype(str).str.upper().str.strip() == "AMAZON", "Amazon", "Otros")

company_core_mean = df.groupby("company_v")[available_core].mean().round(2)
company_core_n = df.groupby("company_v").size().rename("n")
company_core_summary = company_core_mean.join(company_core_n).sort_values("satis", ascending=False)

display(company_core_summary)

In [ ]:
# Gráfico 6: satisfacción promedio por compañía
order = df.groupby("company_v")["satis"].mean().sort_values(ascending=False).index

plt.figure(figsize=(11, 5))
sns.barplot(data=df, x="company_v", y="satis", order=order, errorbar="se")
plt.title("Satisfacción promedio por compañía")
plt.xlabel("Compañía")
plt.ylabel("Satisfacción promedio")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
# Gráfico 7: Amazon vs otros en indicadores globales
amazon_vs_others = df.groupby("is_amazon")[available_core].mean().T.reset_index()
amazon_vs_others.columns.name = None
amazon_vs_others = amazon_vs_others.rename(columns={"index": "variable"})

display(amazon_vs_others)

plot_data = amazon_vs_others.melt(id_vars="variable", var_name="grupo", value_name="promedio")

plt.figure(figsize=(12, 5))
sns.barplot(data=plot_data, x="variable", y="promedio", hue="grupo")
plt.title("Amazon vs otros competidores: indicadores globales")
plt.xlabel("Variable")
plt.ylabel("Promedio")
plt.xticks(rotation=45, ha="right")
plt.legend(title="Grupo")
plt.tight_layout()
plt.show()

In [ ]:
# Gráfico 8: distribución de satisfacción para Amazon vs otros
plt.figure(figsize=(9, 5))
sns.kdeplot(data=df, x="satis", hue="is_amazon", fill=True, common_norm=False, alpha=0.35)
plt.title("Distribución de satisfacción: Amazon vs otros")
plt.xlabel("Satisfacción")
plt.ylabel("Densidad")
plt.tight_layout()
plt.show()

## 8. Matriz de dimensiones de experiencia

Construimos dimensiones agregadas.  
Cada dimensión representa un promedio de variables relacionadas con una parte de la experiencia de compra.  
Esto ayuda a pasar de muchas preguntas de encuesta a una lectura estratégica más clara.


In [ ]:
# ============================================================
# 8. CONSTRUCCIÓN DE DIMENSIONES GERENCIALES
# ============================================================

for dim, cols in experience_dimensions.items():
    valid_cols = [c for c in cols if c in df.columns]
    if valid_cols:
        df[dim] = df[valid_cols].mean(axis=1)

strategic_dims = list(experience_dimensions.keys())
strategic_dims = [d for d in strategic_dims if d in df.columns]

# Resumen de dimensiones
summary_dims = df.groupby("is_amazon")[strategic_dims].mean().T.round(2)
summary_dims["Brecha Amazon - Otros"] = (summary_dims.get("Amazon") - summary_dims.get("Otros")).round(2)

display(summary_dims)

In [ ]:
# Gráfico 9: dimensiones estratégicas Amazon vs otros
summary_plot = df.groupby("is_amazon")[strategic_dims].mean().reset_index()
summary_long = summary_plot.melt(id_vars="is_amazon", var_name="dimension", value_name="promedio")

plt.figure(figsize=(12, 6))
sns.barplot(data=summary_long, y="dimension", x="promedio", hue="is_amazon")
plt.title("Dimensiones de experiencia: Amazon vs otros")
plt.xlabel("Promedio")
plt.ylabel("Dimensión")
plt.legend(title="Grupo")
plt.tight_layout()
plt.show()

In [ ]:
# Gráfico 10: radar chart para Amazon vs otros
# Nota: radar chart es útil para discusión gerencial, aunque no sustituye una tabla precisa.

radar_data = df.groupby("is_amazon")[strategic_dims].mean()

labels = strategic_dims
angles = np.linspace(0, 2 * np.pi, len(labels), endpoint=False).tolist()
angles += angles[:1]

plt.figure(figsize=(8, 8))
ax = plt.subplot(111, polar=True)

for grupo in radar_data.index:
    values = radar_data.loc[grupo, labels].tolist()
    values += values[:1]
    ax.plot(angles, values, linewidth=2, label=grupo)
    ax.fill(angles, values, alpha=0.08)

ax.set_xticks(angles[:-1])
ax.set_xticklabels(labels, fontsize=9)
ax.set_title("Perfil de experiencia por dimensiones", y=1.08)
ax.set_ylim(0, 10)
ax.legend(loc="upper right", bbox_to_anchor=(1.25, 1.10))
plt.tight_layout()
plt.show()

## 9. Heatmap competitivo por compañía

El heatmap resume fortalezas y debilidades por compañía.  
Este gráfico es útil para gerencia porque permite ver dónde Amazon está por encima o por debajo de la categoría.


In [ ]:
# ============================================================
# 9. HEATMAP COMPETITIVO
# ============================================================

company_dims = df.groupby("company_v")[strategic_dims].mean()

plt.figure(figsize=(12, 7))
sns.heatmap(company_dims, annot=True, fmt=".1f", cmap="YlGnBu", linewidths=0.5)
plt.title("Mapa competitivo de dimensiones de experiencia")
plt.xlabel("Dimensión")
plt.ylabel("Compañía")
plt.xticks(rotation=35, ha="right")
plt.tight_layout()
plt.show()

## 10. Correlaciones exploratorias

Las correlaciones no prueban causalidad, pero ayudan a encontrar asociaciones que pueden orientar decisiones.  
Aquí buscamos qué dimensiones se mueven más cerca de satisfacción, recompra y recomendación.


In [ ]:
# ============================================================
# 10. CORRELACIONES EXPLORATORIAS
# ============================================================

corr_vars = strategic_dims + ["satis", "repur", "recomm", "poverq", "soverq", "pq"]
corr_vars = list(dict.fromkeys([c for c in corr_vars if c in df.columns]))

corr_matrix = df[corr_vars].corr(method="spearman")

plt.figure(figsize=(12, 9))
sns.heatmap(corr_matrix, annot=True, fmt=".2f", cmap="coolwarm", center=0, linewidths=0.4)
plt.title("Correlaciones exploratorias entre dimensiones e indicadores globales")
plt.tight_layout()
plt.show()

In [ ]:
# Tabla de asociaciones con satisfacción
corr_with_satis = corr_matrix["satis"].drop("satis").sort_values(ascending=False).reset_index()
corr_with_satis.columns = ["variable", "correlación_spearman_con_satisfacción"]
display(corr_with_satis)

# Gráfico 11: asociación con satisfacción
plt.figure(figsize=(9, 5))
sns.barplot(data=corr_with_satis, y="variable", x="correlación_spearman_con_satisfacción")
plt.title("Variables más asociadas con satisfacción")
plt.xlabel("Correlación de Spearman")
plt.ylabel("Variable")
plt.tight_layout()
plt.show()

## 11. Análisis de brechas: importancia aproximada vs desempeño

Una forma simple de priorizar oportunidades es combinar dos elementos:

1. **Desempeño de Amazon** en cada dimensión.
2. **Asociación de la dimensión con satisfacción**.

No es un modelo predictivo. Es una matriz exploratoria para abrir discusión estratégica.


In [ ]:
# ============================================================
# 11. MATRIZ EXPLORATORIA DE PRIORIDADES
# ============================================================

amazon_df = df[df["is_amazon"] == "Amazon"].copy()
others_df = df[df["is_amazon"] == "Otros"].copy()

priority_rows = []
for dim in strategic_dims:
    amazon_mean = amazon_df[dim].mean()
    others_mean = others_df[dim].mean()
    gap = amazon_mean - others_mean
    corr_satis = df[[dim, "satis"]].corr(method="spearman").iloc[0, 1]
    priority_rows.append({
        "dimension": dim,
        "amazon_mean": amazon_mean,
        "others_mean": others_mean,
        "gap_amazon_minus_others": gap,
        "association_with_satisfaction": corr_satis
    })

priority = pd.DataFrame(priority_rows)
priority["amazon_mean"] = priority["amazon_mean"].round(2)
priority["others_mean"] = priority["others_mean"].round(2)
priority["gap_amazon_minus_others"] = priority["gap_amazon_minus_others"].round(2)
priority["association_with_satisfaction"] = priority["association_with_satisfaction"].round(2)

# Regla interpretativa simple
priority["lectura_gerencial"] = np.where(
    (priority["gap_amazon_minus_others"] < 0) & (priority["association_with_satisfaction"] > priority["association_with_satisfaction"].median()),
    "Prioridad alta: brecha negativa en dimensión relevante",
    np.where(
        priority["gap_amazon_minus_others"] < 0,
        "Revisar: brecha negativa",
        "Fortaleza o posición favorable"
    )
)

display(priority.sort_values(["lectura_gerencial", "gap_amazon_minus_others"]))

In [ ]:
# Gráfico 12: matriz desempeño vs asociación con satisfacción
plt.figure(figsize=(10, 6))
sns.scatterplot(
    data=priority,
    x="amazon_mean",
    y="association_with_satisfaction",
    size=abs(priority["gap_amazon_minus_others"]),
    sizes=(80, 400),
    hue="gap_amazon_minus_others",
    palette="coolwarm",
    legend=True
)

for _, row in priority.iterrows():
    plt.text(row["amazon_mean"] + 0.03, row["association_with_satisfaction"] + 0.005, row["dimension"], fontsize=8)

plt.axvline(priority["amazon_mean"].median(), linestyle="--", linewidth=1)
plt.axhline(priority["association_with_satisfaction"].median(), linestyle="--", linewidth=1)
plt.title("Matriz exploratoria: desempeño de Amazon vs asociación con satisfacción")
plt.xlabel("Desempeño promedio de Amazon")
plt.ylabel("Asociación con satisfacción")
plt.tight_layout()
plt.show()

## 12. Distribuciones por dimensión: Amazon vs otros

Los promedios pueden ocultar heterogeneidad.  
Por eso usamos violines o boxplots para ver dispersión, concentración y presencia de clientes con experiencias extremas.


In [ ]:
# ============================================================
# 12. DISTRIBUCIONES POR DIMENSIÓN
# ============================================================

for dim in strategic_dims:
    plt.figure(figsize=(9, 4.5))
    sns.violinplot(data=df, x="is_amazon", y=dim, inner="quartile", cut=0)
    plt.title(f"Distribución de {dim}: Amazon vs otros")
    plt.xlabel("Grupo")
    plt.ylabel("Puntaje")
    plt.tight_layout()
    plt.show()

## 13. Perfil de cliente y comportamiento de compra

La minería exploratoria también debe responder quiénes están evaluando la marca y cómo compran.  
Estos gráficos ayudan a discutir si las diferencias de satisfacción pueden estar relacionadas con composición de usuarios, canal, método de pago o frecuencia de compra.


In [ ]:
# ============================================================
# 13. PERFIL DE CLIENTE Y COMPORTAMIENTO
# ============================================================

# Diccionarios de etiquetas para variables categóricas principales
labels = {
    "gender": {1: "Male", 2: "Female"},
    "VN_1009_TP20": {1: "Mobile App", 2: "Website PC", 3: "Mobile Website"},
    "VN_1009_TP21": {1: "Credit Cards", 2: "PayPal", 3: "E-nets", 4: "AXS machines", 5: "Cash on delivery", 6: "Others"},
    "VN_1009_TP24_1": {1: "Yes", 2: "No"},
    "VN_1009_TP24_2": {1: "Yes", 2: "No"},
    "VN_1009_TP25A": {1: "Physical store", 2: "Online store", 3: "Equal both"}
}

plot_df = df.copy()
for col, mapping in labels.items():
    if col in plot_df.columns:
        plot_df[col + "_label"] = plot_df[col].map(mapping)

print("Etiquetas creadas para variables categóricas.")

In [ ]:
# Gráfico 13: edad por grupo
plt.figure(figsize=(9, 5))
sns.histplot(data=df, x="age", hue="is_amazon", bins=20, kde=True, common_norm=False)
plt.title("Distribución de edad: Amazon vs otros")
plt.xlabel("Edad")
plt.ylabel("Frecuencia")
plt.tight_layout()
plt.show()

In [ ]:
# Gráfico 14: canal de compra más frecuente por grupo
if "VN_1009_TP20_label" in plot_df.columns:
    channel_table = pd.crosstab(plot_df["VN_1009_TP20_label"], plot_df["is_amazon"], normalize="columns") * 100
    display(channel_table.round(1))

    channel_plot = channel_table.reset_index().melt(id_vars="VN_1009_TP20_label", var_name="grupo", value_name="porcentaje")
    plt.figure(figsize=(9, 5))
    sns.barplot(data=channel_plot, x="VN_1009_TP20_label", y="porcentaje", hue="grupo")
    plt.title("Canal de compra más frecuente")
    plt.xlabel("Canal")
    plt.ylabel("Porcentaje dentro del grupo")
    plt.xticks(rotation=30, ha="right")
    plt.tight_layout()
    plt.show()

In [ ]:
# Gráfico 15: método de pago preferido por grupo
if "VN_1009_TP21_label" in plot_df.columns:
    pay_table = pd.crosstab(plot_df["VN_1009_TP21_label"], plot_df["is_amazon"], normalize="columns") * 100
    display(pay_table.round(1))

    pay_plot = pay_table.reset_index().melt(id_vars="VN_1009_TP21_label", var_name="grupo", value_name="porcentaje")
    plt.figure(figsize=(10, 5))
    sns.barplot(data=pay_plot, x="VN_1009_TP21_label", y="porcentaje", hue="grupo")
    plt.title("Método de pago preferido")
    plt.xlabel("Método de pago")
    plt.ylabel("Porcentaje dentro del grupo")
    plt.xticks(rotation=35, ha="right")
    plt.tight_layout()
    plt.show()

In [ ]:
# Gráfico 16: gasto promedio por visita y frecuencia de compra
fig_data = df[["is_amazon", "Q9C_P", "Q9D"]].dropna()

plt.figure(figsize=(9, 5))
sns.boxplot(data=fig_data, x="is_amazon", y="Q9C_P")
plt.title("Número de compras en los últimos 6 meses")
plt.xlabel("Grupo")
plt.ylabel("Compras")
plt.tight_layout()
plt.show()

plt.figure(figsize=(9, 5))
sns.boxplot(data=fig_data, x="is_amazon", y="Q9D")
plt.title("Monto promedio gastado por visita")
plt.xlabel("Grupo")
plt.ylabel("Monto promedio")
plt.tight_layout()
plt.show()

## 14. Atípicos: ¿ruido o señal?

En la presentación, los atípicos no se eliminan automáticamente.  
Pueden ser errores de captura, pero también pueden representar clientes extremadamente insatisfechos o usuarios muy intensivos.  
Aquí los identificamos visualmente para discutir si son ruido o señal.


In [ ]:
# ============================================================
# 14. ATÍPICOS EXPLORATORIOS
# ============================================================

outlier_vars = ["Q9C_P", "Q9D", "age", "satis", "repur", "recomm"]
outlier_vars = [c for c in outlier_vars if c in df.columns]

for col in outlier_vars:
    plt.figure(figsize=(8, 3.8))
    sns.boxplot(data=df, x=col)
    plt.title(f"Revisión de atípicos: {col}")
    plt.xlabel(col)
    plt.tight_layout()
    plt.show()

In [ ]:
# Tabla de registros extremos usando regla IQR para variables continuas seleccionadas
extreme_summary = []
for col in ["Q9C_P", "Q9D", "age"]:
    if col in df.columns:
        q1 = df[col].quantile(0.25)
        q3 = df[col].quantile(0.75)
        iqr = q3 - q1
        lower = q1 - 1.5 * iqr
        upper = q3 + 1.5 * iqr
        n_extreme = ((df[col] < lower) | (df[col] > upper)).sum()
        extreme_summary.append({
            "variable": col,
            "q1": q1,
            "q3": q3,
            "lower_limit": lower,
            "upper_limit": upper,
            "n_extreme": int(n_extreme),
            "pct_extreme": round(n_extreme / len(df) * 100, 2)
        })

extreme_summary = pd.DataFrame(extreme_summary)
display(extreme_summary)

## 15. Matriz gerencial final del EDA

Esta tabla resume lo encontrado en términos accionables.  
No reemplaza el criterio del gerente, pero organiza la evidencia para discusión.


In [ ]:
# ============================================================
# 15. MATRIZ GERENCIAL FINAL
# ============================================================

managerial_matrix = priority.copy()
managerial_matrix["pregunta_para_discusión"] = managerial_matrix.apply(
    lambda r: f"¿Qué acción debería tomar Amazon en {r['dimension']} si su brecha es {r['gap_amazon_minus_others']} y su asociación con satisfacción es {r['association_with_satisfaction']}?",
    axis=1
)

managerial_matrix = managerial_matrix.sort_values(
    by=["association_with_satisfaction", "gap_amazon_minus_others"],
    ascending=[False, True]
)

display(managerial_matrix)

In [ ]:
# Gráfico 17: ranking de prioridades por asociación con satisfacción
plt.figure(figsize=(10, 5))
sns.barplot(data=managerial_matrix, y="dimension", x="association_with_satisfaction")
plt.title("Ranking exploratorio de dimensiones asociadas con satisfacción")
plt.xlabel("Asociación con satisfacción")
plt.ylabel("Dimensión")
plt.tight_layout()
plt.show()

## 16. Actividad de cierre para clase

### Discusión en grupos

Cada grupo debe seleccionar una dimensión de experiencia y responder:

1. ¿Amazon está mejor o peor que el mercado en esa dimensión?
2. ¿La dimensión parece relevante para satisfacción, recompra o recomendación?
3. ¿Qué gráfico apoya mejor la conclusión?
4. ¿Qué decisión gerencial se podría tomar antes de construir un modelo predictivo?
5. ¿Qué dato adicional sería necesario para tomar una mejor decisión?

---

## Conclusión metodológica

La minería de datos no inicia con Machine Learning.  
Primero se necesita construir una lectura confiable de la base: calidad de datos, patrones, brechas, atípicos, comportamiento del cliente y prioridades gerenciales.

Este EDA deja el terreno listo para una segunda etapa, donde sí podrían aparecer modelos predictivos o segmentación avanzada, pero solo después de tener claro el problema de negocio y la estructura real de los datos.
